## tl;dr
Exposure normalization before stereo barely changes this fixed interior-patch NCC proxy. Across 248 directed train-camera pairs, median paired improvement is 0.000369 for ingest cancellation and 0.000637 for scalar correction, below the pre-recorded 0.005 gate. Neither variant is promoted to dense reconstruction; the surface-repair goal remains incomplete.

## Context & Methods
2026-09-06, clever-shadow, frame 000973. All 62 train cameras; four nearest neighbors by camera-center geometry. Ordinary grayscale NCC on the same fully visible mesh-warped 11x11 patches. Original EXRs generate JPEG98 4:4:4 inputs with original gains, one geometric-mean train gain, or original gain times the existing train-only scalar correction. All 62 original JPEG replays are byte-identical.

### Key Assumptions
This is **not** an exact replay of COLMAP's bilateral CUDA photometric cost. Fully visible current-mesh interior patches exclude silhouettes and occlusion boundaries. Patch observations are correlated within a camera pair; pair-block medians are the primary gate. The scalar correction was fitted on train data: this is a paired diagnostic, not an untouched generalization test. No held-camera RGB or semantic masks are read, and no new face-quality or full-frame metric is computed.

[Upstream photometric cost](https://github.com/colmap/colmap/blob/5509fffe/src/colmap/mvs/patch_match_cuda.cu).

## Data
Artifacts: `/mnt/data/lookcloser_dec5_5a3_surface_repair/stereo_input_exposure_control`. Requires NumPy/pandas and the retained filesystem. The separate integrity audit reprojects 24 patches directly from source images; maximum discrepancy is zero.

In [1]:
from pathlib import Path
import hashlib,json
import numpy as np
import pandas as pd
root=Path('/mnt/data/lookcloser_dec5_5a3_surface_repair/stereo_input_exposure_control')
def read(path): return json.loads(Path(path).read_text())
def digest(path):
    value=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(8<<20),b''): value.update(block)
    return value.hexdigest()
audit=read(root/'integrity_audit.json')
for path,expected in audit['retained_hashes'].items(): assert digest(path)==expected,path
findings=read(root/'findings.json');request=read(root/'request.json')
assert digest(root/'request.json')==findings['request_sha256']
assert not request['uses_eval_rgb'] and not request['uses_semantic_masks']
assert audit['maximum_spotcheck_error']==0 and len(audit['spotchecks'])==24
print('Verified hashes:',len(audit['retained_hashes']))

Verified hashes: 260


## Results
### Identical inventory and paired summaries

In [2]:
samples=np.load(root/'observations.npz',allow_pickle=False)['rows']
assert samples.shape==(165404,10) and np.isfinite(samples).all()
assert len(np.unique(samples[:,:4],axis=0))==len(samples)
methods=audit['methods'];pair_keys=np.unique(samples[:,:2],axis=0)
assert len(pair_keys)==248
values=[]
for i,j in pair_keys:
    rows=samples[(samples[:,0]==i)&(samples[:,1]==j)]
    values.append(np.median(rows[:,4:7],axis=0))
values=np.asarray(values);table=[]
for k,method in enumerate(methods):
    delta=values[:,k]-values[:,0]
    saved=findings['summaries'][k]
    assert saved['median_paired_improvement']==np.median(delta)
    assert saved['pair_median_ncc']==np.median(values[:,k])
    table.append(saved)
display(pd.DataFrame(table))
assert not any(row['eligible_for_dense_control'] for row in table)

,eligible_for_dense_control,improved_pair_fraction,median_paired_improvement,method,pair_median_ncc
0,False,0.000000,0.000000,original,0.683486
1,False,0.540323,0.000369,cancel_ingest,0.685846
2,False,0.572581,0.000637,scalar_fit,0.686020


### Contrast strata and inspected input examples
Contrast bins are defined from the original reference patch only, not changed per candidate. These are not semantic skin regions.

In [3]:
display(pd.DataFrame(audit['reference_contrast_strata']))
assert sum(r['patch_pairs'] for r in audit['reference_contrast_strata'])==len(samples)
review=read(root/'visual_review.json')
assert not review['uses_eval_rgb'] and not review['new_reconstruction_rendered']
for row in review['rows']:
    crop=Path(row['crop']); assert crop.is_file()
    inputs=read(crop.parent/'review_inputs.json')
    for item in inputs['images']: assert digest(item['path'])==item['sha256']
display(pd.DataFrame(review['rows'])[['physical_camera','status','notes']])

,fraction_ncc_at_least_point_one,median_ncc,median_paired_delta,patch_pairs,reference_gray_std_range
0,"[0.9354535695115406, 0.9358561460010736, 0.935...","[0.577431160720848, 0.579898135946123, 0.58206...","[0.0, 0.0013757010228334365, 0.002784307405035...",7452,"[0, 0.01]"
1,"[0.9547724164006983, 0.954961467032906, 0.9549...","[0.7310147244283143, 0.7312291234952998, 0.731...","[0.0, 0.00019439484664296458, 0.00024728723458...",89923,"[0.01, 0.03]"
2,"[0.913698098564222, 0.9137757081878153, 0.9141...","[0.632963060648762, 0.6341647518170942, 0.6335...","[0.0, 0.00026603653906348557, 0.00022178644812...",64425,"[0.03, 0.1]"
3,"[0.9500554938956715, 0.9508879023307436, 0.950...","[0.9160345322458916, 0.9164908643817266, 0.916...","[0.0, 0.00013691264823662008, 8.13679114018417...",3604,"[0.1, 1]"


,physical_camera,status,notes
0,E004_C005_1210YM,inspected_input_only,Cancel-ingest and scalar-fit darken the hand a...
1,E004_B005_1210I7,inspected_input_only,"All three inputs retain the same hand, tube an..."
2,G004_B005_1210FG,inspected_input_only,The corrections reduce brightness slightly whi...


## Takeaways
The known exposure mismatch changes source brightness, but is not a strong NCC driver on this measured interior pool. This result does not rule out effects at object boundaries or prove that the geometry is correct. No new PatchMatch job, reconstructed render, face metric, temporal validation or fly-through pass is claimed. Inspect the three train-input comparison sheets and the separate surface-repair report for scope and next experiments.